In [ ]:
!pip install -q openai torch numpy datasets transformers

In [2]:
import openai
from openai import OpenAI
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
import re
import time
from collections import defaultdict
import numpy as np
from dataclasses import dataclass
from typing import List, Dict, Tuple

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
)

import warnings

warnings.filterwarnings("ignore")

os.environ["OPENAI_API_KEY"] = ""
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

print(f"Using device: {DEVICE}")
client = openai.OpenAI()
# Frozen math-pretrained backbone; only the small reward heads are trained
BACKBONE_NAME = "Qwen/Qwen2.5-Math-1.5B"
SEED = 0
torch.manual_seed(SEED)

Using device: mps


# 1) Reward Models

Why Do we need?

- After SFT, a model can follow instructions — but it has no sense of which of its possible outputs is best.
- Given a math problem, it might generate ten different solutions: some correct, some wrong, some with flawed reasoning that happens to reach the right answer.
- SFT alone can't distinguish these because it only learned to mimic training data, not to evaluate quality.
- GRPO doesn't replace reward models — they do different jobs. GRPO is the optimization algorithm (how to update weights), ORM/PRM are the scoring functions (how good is each output). GRPO generates multiple responses and reinforces the better ones, but it needs scores to rank them. For math, a simple rule works (check the answer). For open-ended tasks like summarization, you need a learned reward model to provide those scores.
- PRM also gives something GRPO can't: step-level feedback. GRPO only compares full responses, but a PRM scores each reasoning step individually — pinpointing exactly where the logic broke instead of penalizing the entire response.



In reinforcement learning for LLMs, the central challenge is:
> **Which tokens / steps are responsible for a correct or incorrect answer?**

Consider a multi-step math solution with 8 steps where step 3 is wrong.

<img src="https://www.marktechpost.com/wp-content/uploads/2025/01/Screenshot-2025-01-19-at-11.30.26%E2%80%AFAM.png" width=800>


Process Reward Models (PRMs) and Outcome Reward Models (ORMs) differ in how they evaluate LLM generations:

**Outcome Reward Model (ORM)**:
- Looks only at the final answer
- Assigns reward = 1 if correct, 0 if wrong
- ALL 8 steps get the same sparse signal
- Cannot identify *where* the error occurred

**Process Reward Model (PRM)**:
- Scores each individual step
- Step 3 gets a low score; steps 1,2 get high scores
- Credit is correctly assigned — dense signal

# 2) Outcome Reward Model (ORM)

An ORM maps a (problem, full_solution) pair to a scalar reward.
It only evaluates the quality of the *outcome* — not the reasoning process.

**Architecture:**
1. Load a pretrained LM backbone (e.g., DistilBERT, LLaMA, GPT-2)
2. Remove the original language modeling head
3. Add a scalar regression head on top of the [EOS] / last token
4. Train with Bradley-Terry loss (pairwise) or BCE loss (binary)

In [3]:
def load_backbone(model_name: str = BACKBONE_NAME):
    """
    Load a frozen pretrained decoder backbone (bf16) and its tokenizer (right padding).
    """
    print(f"Loading backbone: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="right")
    backbone = AutoModel.from_pretrained(model_name, dtype=torch.bfloat16)
    hidden_dim = backbone.config.hidden_size

    # Add pad token if not present
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"  hidden_dim = {hidden_dim}")
    print(f"  vocab_size  = {tokenizer.vocab_size}")
    return tokenizer, backbone, hidden_dim


tokenizer, backbone, HIDDEN_DIM = load_backbone()
backbone = backbone.to(DEVICE).eval()

Loading backbone: Qwen/Qwen2.5-Math-1.5B


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  hidden_dim = 1536
  vocab_size  = 151643


## 2.1 Model

The Outcome Reward Model takes a complete (problem, solution) pair and outputs a single scalar reward score. It is used to judge whether a full reasoning chain is correct.

Architecture:
1. `[CLS] problem [SEP] solution_step_1 ... solution_step_K [EOS]`
2. Transformer backbone (frozen or fine-tuned)
3. Hidden state at the last token (in a causal backbone, the only position that has read the
   whole problem and solution)
4. Scalar regression head (2-layer MLP)
5. Reward ∈ ℝ (unbounded; higher = more correct)

In [4]:
# === Train a Reward Model ===
class OutcomeRewardModel(nn.Module):
    """
    Outcome Reward Model (ORM).
    Takes a complete (problem, solution) pair and outputs a single scalar reward.
    """

    def __init__(self, backbone: nn.Module, hidden_dim: int, dropout_p: float = 0.1):
        super().__init__()
        self.backbone = backbone

        # Scalar reward head: 2-layer MLP with GELU activation
        # (batch_num, hidden_dim) → (batch_num, 1)
        self.reward_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim // 2, 1),
        )

        # Initialize reward head weights to small values
        nn.init.normal_(self.reward_head[0].weight, std=0.02)
        nn.init.zeros_(self.reward_head[0].bias)
        nn.init.normal_(self.reward_head[-1].weight, std=0.02)
        nn.init.zeros_(self.reward_head[-1].bias)

    def forward(
        self, input_ids: torch.Tensor, attention_mask: torch.Tensor
    ) -> torch.Tensor:
        """
        Forward pass: encode (problem + solution) → scalar reward.
        """

        # (batch_num, seq_len) → (batch_num, seq_len, hidden_dim)
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state.float()

        # Causal backbone: pool the LAST real token, the only one that has read everything
        # (batch_num,) index of the last non-pad token (right padding)
        last_idx = attention_mask.sum(dim=1) - 1
        # (batch_num, seq_len, hidden_dim) → (batch_num, hidden_dim)
        pooled = last_hidden[
            torch.arange(last_hidden.size(0), device=last_hidden.device), last_idx
        ]

        # (batch_num, hidden_dim) → (batch_num, 1)
        reward_logit = self.reward_head(pooled)

        # Squeeze last dim → scalar reward per example
        # (batch_num, 1) → (batch_num,)
        return reward_logit.squeeze(-1)


orm = OutcomeRewardModel(backbone, HIDDEN_DIM).to(DEVICE)
orm_params = sum(p.numel() for p in orm.reward_head.parameters())
print(f"\nORM reward head params: {orm_params:,}")
print(f"ORM total params: {sum(p.numel() for p in orm.parameters()):,}")


ORM reward head params: 1,181,185
ORM total params: 1,544,895,489


## 2.2 Loss Functions
**Bradley-Terry Pairwise Loss** (standard RLHF):
$$
    L_BT = −log σ(r(x, y_chosen) − r(x, y_rejected))
$$

**Binary Cross-Entropy Loss** (for verified labels):
$$
    L_BCE = −[y * log(σ(r)) + (1−y) * log(1−σ(r))]
$$


**Margin-Ranking Loss** (optional regularization):
$$
    L_margin = max(0, m − (r_chosen − r_rejected))
$$

In [5]:
def bradley_terry_loss(
    r_chosen: torch.Tensor, r_rejected: torch.Tensor
) -> torch.Tensor:
    """
    Bradley-Terry pairwise ranking loss.
    L = log(1 + exp(r_r − r_c)) = softplus(r_r − r_c)
    """
    # (batch_num,) and (batch_num,) → (batch_num,)
    margin = r_chosen - r_rejected

    # (batch_num,) → (batch_num,)
    loss = F.softplus(-margin)

    # (batch_num,) → scalar
    return loss.mean()


def bce_reward_loss(rewards: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """
    Binary Cross-Entropy loss for ORM with verified labels.
    """
    # (batch_num,) and (batch_num,) → scalar
    return F.binary_cross_entropy_with_logits(rewards, labels)


def margin_ranking_loss(
    r_chosen: torch.Tensor, r_rejected: torch.Tensor, margin: float = 0.5
):
    """
    Margin ranking loss: enforces r_chosen > r_rejected by at least margin.
    """
    # (batch_num,) → (batch_num,)
    violation = margin - (r_chosen - r_rejected)

    # (batch_num,) → (batch_num,)
    loss = torch.clamp(violation, min=0.0)
    return loss.mean()

## 2.3 Training Dataset

Each training example is a **preference pair**:
    `(problem, chosen_solution, rejected_solution)`

The pairs come from **Math-Shepherd** (Wang et al., 2024): GSM8K problems with several sampled
solutions each, labelled step by step by Monte Carlo rollouts. A solution whose every step is
`+` reaches the right answer; pairing it with a solution of the same problem that contains a `-`
gives a chosen/rejected pair. Problems are split 80/20 so test pairs come from unseen problems.

Where:
- `chosen_solution`: a solution that arrives at the CORRECT final answer
- `rejected_solution`: a solution that arrives at the WRONG final answer

The ORM never sees intermediate steps — it only sees the complete solution.

In [6]:
@dataclass
class ORMExample:
    problem: str
    chosen_solution: str
    rejected_solution: str
    correct_answer: str


@torch.no_grad()
def encode_solution(
    problem: str, steps: List[str], max_len: int = 512
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Frozen-backbone features of one solution: the last-token vector (hidden_dim,) and the
    last-token vector of every step (num_steps, hidden_dim). Steps are newline-separated; the
    tokenizer's character offsets locate the token where each step ends."""
    text = problem + "\n" + "\n".join(steps)
    enc = tokenizer(
        text,
        return_offsets_mapping=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
    )
    # (seq_len, 2) character span of every token
    offsets = enc.pop("offset_mapping")[0]
    # (seq_len, hidden_dim)
    hidden = backbone(**enc.to(DEVICE)).last_hidden_state[0].float().cpu()
    ends, pos = [], len(problem) + 1
    for step in steps:
        pos += len(step)
        ends.append(pos)
        pos += 1
    # last token that starts before the step's final character
    step_pos = [
        min(int((offsets[:, 0] < end).sum()) - 1, hidden.size(0) - 1) for end in ends
    ]
    return hidden[-1], hidden[step_pos]

In [7]:
def parse_shepherd(row: dict):
    """(problem, steps, step_labels) from one Math-Shepherd row, or None if it does not parse.

    'input' ends every step with the marker 'ки'; 'label' is the same text with '+' (a rollout
    from this step can still reach the right answer) or '-' in its place.
    """
    problem, sep, rest = row["input"].partition(" Step 1:")
    if not sep:
        return None
    steps = [s.strip() for s in ("Step 1:" + rest).split("ки") if s.strip()]
    labels = re.findall(r" ([+-])(?=\nStep \d+:|\s*$)", row["label"])
    if len(labels) != len(steps):
        return None
    return problem.strip(), steps, [int(label == "+") for label in labels]


start = time.time()
# problem -> list of (steps, labels)
solutions = defaultdict(list)
# Rows are stored grouped by task and by final label, so shuffle before sampling
shepherd = load_dataset("peiyi9979/Math-Shepherd", split="train")
shepherd = shepherd.filter(lambda task: task == "GSM8K", input_columns="task")
for row in shepherd.shuffle(seed=SEED).select(range(30_000)):
    parsed = parse_shepherd(row)
    if parsed is not None:
        solutions[parsed[0]].append((parsed[1], parsed[2]))

problems = sorted(solutions)
random.Random(SEED).shuffle(problems)
split = int(0.8 * len(problems))
train_problems, test_problems = problems[:split], problems[split:]


def make_pairs(problem_list: List[str], max_pairs: int) -> List[ORMExample]:
    """One (all-correct, contains-an-error) pair per problem that has both."""
    pairs = []
    for p in problem_list:
        good = [s for s, l in solutions[p] if all(l)]
        bad = [s for s, l in solutions[p] if not all(l)]
        if good and bad:
            pairs.append(ORMExample(p, "\n".join(good[0]), "\n".join(bad[0]), ""))
        if len(pairs) == max_pairs:
            break
    return pairs


train_pairs, test_pairs = make_pairs(train_problems, 400), make_pairs(
    test_problems, 100
)
print(
    f"{len(problems):,} GSM8K problems, {sum(len(v) for v in solutions.values()):,} solutions "
    f"({time.time() - start:.0f}s)"
)
print(f"ORM preference pairs: {len(train_pairs)} train, {len(test_pairs)} test")
print(f"\nExample problem: {train_pairs[0].problem[:120]}...")
print(f"Chosen:   {train_pairs[0].chosen_solution[:150]}...")
print(f"Rejected: {train_pairs[0].rejected_solution[:150]}...")

7,295 GSM8K problems, 29,986 solutions (4s)
ORM preference pairs: 400 train, 100 test

Example problem: Diana wants to buy winter clothes for all the 40 children at her local children's home. The home has five times as many ...
Chosen:   Step 1: If the home has five times as many teenagers as toddlers, then it has 5*6 = <<5*6=30>>30 teenagers.
Step 2: Adding the teenagers and toddlers,...
Rejected: Step 1: If there are 6 toddlers, the home has 6 x 5 = <<6*5=30>>30 teenagers.
Step 2: The total number of children is therefore 6 + 30 + 40 = <<6+30+4...


## 2.4 Training

In [8]:
def train_orm(
    model: OutcomeRewardModel,
    train_feats: Tuple[torch.Tensor, torch.Tensor],
    test_feats: Tuple[torch.Tensor, torch.Tensor],
    num_epochs: int = 60,
    lr: float = 1e-3,
    batch_num: int = 64,
    margin: float = 0.0,
) -> Tuple[List[float], List[float]]:
    """Train only the reward head, on cached last-token features of (chosen, rejected).

    The backbone is frozen, so its features are computed once; each epoch costs milliseconds.
    """
    head = model.reward_head
    optimizer = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=0.01)
    chosen, rejected = (t.to(DEVICE) for t in train_feats)
    test_chosen, test_rejected = (t.to(DEVICE) for t in test_feats)
    loss_history, accuracy_history = [], []

    for epoch in range(num_epochs):
        head.train()
        perm = torch.randperm(chosen.size(0), device=DEVICE)
        epoch_losses = []
        for i in range(0, len(perm), batch_num):
            idx = perm[i : i + batch_num]
            # (batch_num, hidden_dim) → (batch_num,)
            r_chosen, r_rejected = head(chosen[idx]).squeeze(-1), head(
                rejected[idx]
            ).squeeze(-1)
            loss = bradley_terry_loss(r_chosen, r_rejected)
            if margin > 0:
                loss = loss + 0.1 * margin_ranking_loss(
                    r_chosen, r_rejected, margin=margin
                )
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())

        head.eval()
        with torch.no_grad():
            train_acc = (head(chosen) > head(rejected)).float().mean().item()
            test_acc = (head(test_chosen) > head(test_rejected)).float().mean().item()
        loss_history.append(np.mean(epoch_losses))
        accuracy_history.append(test_acc)
        if epoch % 10 == 0 or epoch == num_epochs - 1:
            print(
                f"  Epoch {epoch+1:3d}/{num_epochs} | BT Loss: {loss_history[-1]:.4f} | "
                f"Ranking Acc train {train_acc:.1%}, test {test_acc:.1%}"
            )
    return loss_history, accuracy_history


# === Features (one frozen forward per solution) and training ===
start = time.time()


def pair_features(pairs: List[ORMExample]) -> Tuple[torch.Tensor, torch.Tensor]:
    chosen = [
        encode_solution(p.problem, p.chosen_solution.split("\n"))[0] for p in pairs
    ]
    rejected = [
        encode_solution(p.problem, p.rejected_solution.split("\n"))[0] for p in pairs
    ]
    return torch.stack(chosen), torch.stack(rejected)


orm_train_feats, orm_test_feats = pair_features(train_pairs), pair_features(test_pairs)
print(
    f"Encoded {2 * (len(train_pairs) + len(test_pairs))} solutions in {time.time() - start:.0f}s"
)
print("Chance ranking accuracy: 50%")
orm_losses, orm_accs = train_orm(orm, orm_train_feats, orm_test_feats)

Encoded 1000 solutions in 160s
Chance ranking accuracy: 50%


  Epoch   1/60 | BT Loss: 0.5579 | Ranking Acc train 82.2%, test 78.0%
  Epoch  11/60 | BT Loss: 0.1255 | Ranking Acc train 99.5%, test 79.0%


  Epoch  21/60 | BT Loss: 0.0291 | Ranking Acc train 100.0%, test 77.0%
  Epoch  31/60 | BT Loss: 0.0078 | Ranking Acc train 100.0%, test 80.0%


  Epoch  41/60 | BT Loss: 0.0031 | Ranking Acc train 100.0%, test 78.0%
  Epoch  51/60 | BT Loss: 0.0021 | Ranking Acc train 100.0%, test 76.0%


  Epoch  60/60 | BT Loss: 0.0011 | Ranking Acc train 100.0%, test 75.0%


## 2.5 Inference

### ORM at Inference Time: Best-of-N (BoN) Selection

The most common use of an ORM at inference time is **Best-of-N**:
1. Sample N solutions from a policy LM for a given problem
2. Score each with the ORM: `r_i = ORM(problem, solution_i)`
3. Return the solution with the highest ORM score

This is also called **reranking** or **weighted voting**.

Formally:
$$
    ŷ = argmax_{i ∈ [N]} r_ORM(x, y_i)
$$

Best-of-N compute scales as O(N) — each solution needs one ORM forward pass.

In [9]:
@torch.no_grad()
def orm_score(
    model: OutcomeRewardModel,
    problem: str,
    solution: str,
    tokenizer,
    max_len: int = 512,
) -> float:
    """
    Score a single (problem, solution) pair with the ORM.
    """
    model.eval()

    # Tokenize (problem, solution) pair
    # Output: dict with (1, seq_len) tensors
    encoded = tokenizer(
        problem,
        solution,
        return_tensors="pt",
        truncation=True,
        max_length=max_len,
        padding="max_length",
    )

    input_ids = encoded["input_ids"].to(DEVICE)
    attention_mask = encoded["attention_mask"].to(DEVICE)

    # Forward: (1, seq_len) → scalar
    score = model(input_ids, attention_mask)

    return score.item()


def best_of_n_orm(
    model: OutcomeRewardModel,
    problem: str,
    solutions: List[str],
    tokenizer,
    verbose: bool = True,
) -> Tuple[str, float, List[float]]:
    """
    Best-of-N selection using ORM scoring.
    """
    all_scores = []
    for i, sol in enumerate(solutions):
        # Score each solution
        score = orm_score(model, problem, sol, tokenizer)
        all_scores.append(score)
        if verbose:
            print(f"  Solution {i+1}: score={score:+.4f} | snippet={sol[:60]!r}")

    # Select best solution by score
    best_idx = int(np.argmax(all_scores))
    best_solution = solutions[best_idx]
    best_score = all_scores[best_idx]

    return best_solution, best_score, all_scores


# Demo ORM inference
test_problem = "A store had 50 apples. They sold 30% Monday and 20% of remainder Tuesday. How many left?"
candidate_solutions = [
    "30% of 50 = 15. After Monday: 35. 20% of 35 = 7. After Tuesday: 28. Answer: 28.",
    "30% of 50 = 15. Tuesday: 20% of 50 = 10. Total sold: 25. Remaining: 25. Answer: 25.",
    "Sold 30% then 20% so total = 50% sold. 50 × 0.5 = 25 remaining. Answer: 25.",
    "Monday: 15 sold, 35 left. Tuesday: 7 sold, 28 left. Answer: 28.",
]
best_sol, best_score, scores = best_of_n_orm(
    orm, test_problem, candidate_solutions, tokenizer
)
print(f"\nBest solution (score={best_score:+.4f}):\n  {best_sol}")

  Solution 1: score=-12.0926 | snippet='30% of 50 = 15. After Monday: 35. 20% of 35 = 7. After Tuesd'


  Solution 2: score=-14.8693 | snippet='30% of 50 = 15. Tuesday: 20% of 50 = 10. Total sold: 25. Rem'


  Solution 3: score=-16.1698 | snippet='Sold 30% then 20% so total = 50% sold. 50 × 0.5 = 25 remaini'


  Solution 4: score=-11.3036 | snippet='Monday: 15 sold, 35 left. Tuesday: 7 sold, 28 left. Answer: '

Best solution (score=-11.3036):
  Monday: 15 sold, 35 left. Tuesday: 7 sold, 28 left. Answer: 28.


# 3) Process Reward Model (PRM)

A PRM assigns a reward to **each individual reasoning step** in a solution.
Instead of waiting until the end, it gives dense, step-level feedback.

**Architecture Difference from ORM**

Both ORM and PRM use the same type of transformer backbone, but differ in:
1. **Input**: PRM takes (problem + steps up to step k) — partial solution
2. **Output**: PRM outputs one score per step, not one per solution
3. **Loss**: PRM uses BCE at each step position independently
4. **Label**: Each step has its own {correct=1, incorrect=0} label

**Step Token Format**

```
Problem: A store...
Step 1: Monday sales = 15. ▪ [LABEL: 1 = correct]
Step 2: After Monday: 35.  ▪ [LABEL: 1 = correct]
Step 3: Tuesday sales = 10. ▪ [LABEL: 0 = WRONG — should be 7]
Step 4: Remaining = 25.    ▪ [LABEL: 0 = wrong]
```

The ▪ token (step separator) is where predictions are made.
The model learns to predict step correctness from this position.

## 3.1 Model

In [10]:
class ProcessRewardModel(nn.Module):
    """
    Process Reward Model (PRM).
    Assigns a correctness probability to each reasoning step.
    """

    def __init__(self, backbone: nn.Module, hidden_dim: int, dropout_p: float = 0.1):
        super().__init__()
        self.backbone = backbone

        # Step-level reward model scoring head: shared across all step positions
        # (num_steps, hidden_dim) → (num_steps, 1)
        self.step_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim // 2, 1),
        )

        for module in self.step_head:
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, std=0.02)
                nn.init.zeros_(module.bias)

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        step_token_positions: torch.Tensor,
    ) -> torch.Tensor:
        """
        Forward pass: encode partial solution → step-level reward scores.
        """
        # (batch_num, seq_len) → (batch_num, seq_len, hidden_dim)
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state.float()

        B = input_ids.size(0)  # batch_num
        K = step_token_positions.size(1)  # num_steps

        # Gather hidden states at each [STEP] token position
        pos_expanded = step_token_positions.unsqueeze(-1).expand(
            B, K, last_hidden.size(-1)
        )

        # Input: (batch_num, seq_len, hidden_dim) and (batch_num, num_steps, hidden_dim)
        # Output: (batch_num, num_steps, hidden_dim)
        step_hidden = torch.gather(last_hidden, dim=1, index=pos_expanded)

        # Reshape for shared step head
        # (batch_num, num_steps, hidden_dim) → (batch_num * num_steps, hidden_dim)
        # take this as CLS for each step
        step_hidden_flat = step_hidden.view(B * K, -1)

        # Apply shared step scoring head
        # (batch_num * num_steps, hidden_dim) → (batch_num * num_steps, 1)
        step_scores_flat = self.step_head(step_hidden_flat)

        # Reshape back
        # (batch_num * num_steps, 1) → (batch_num, num_steps)
        step_scores = step_scores_flat.view(B, K)

        return step_scores


# Instantiate PRM
prm = ProcessRewardModel(backbone, HIDDEN_DIM).to(DEVICE)
prm_head_params = sum(p.numel() for p in prm.step_head.parameters())
print(f"\nPRM step head params: {prm_head_params:,}")


PRM step head params: 1,181,185


## 3.2 Loss Function

The PRM predicts a binary correctness label for each step.
Loss is BCE at each step position, averaged over all steps.

$$
    L_PRM = − (1/K) Σ_{k=1}^{K} [y_k * log σ(r_k) + (1−y_k) * log(1−σ(r_k))]
$$

Where:
    y_k ∈ {0, 1}: label for step k (1=correct, 0=incorrect)
    r_k ∈ ℝ:      PRM logit for step k (before sigmoid)


**Important implementation detail:**

Steps after the FIRST incorrect step should also be labeled as incorrect, even if they are locally correct.

In [11]:
def prm_loss(
    step_scores: torch.Tensor,
    step_labels: torch.Tensor,
    step_mask: torch.Tensor,
    pos_weight: torch.Tensor | None = None,
) -> torch.Tensor:
    """
    Per-step BCE loss for the PRM, averaged over real steps only.
    Padded step slots (step_mask == 0) contribute nothing. pos_weight < 1 down-weights the
    (much more frequent) correct steps so the head does not learn "always correct".
    """
    # (batch_num, max_steps) and (batch_num, max_steps) → (batch_num, max_steps)
    per_step_loss = F.binary_cross_entropy_with_logits(
        step_scores, step_labels, reduction="none", pos_weight=pos_weight
    )

    # Zero out padded steps
    # (batch_num, max_steps) → (batch_num, max_steps)
    per_step_loss = per_step_loss * step_mask

    # (batch_num, max_steps) → scalar
    return per_step_loss.sum() / step_mask.sum()

## 3.3 PRM Training Data

- The biggest challenge for PRMs is **obtaining step-level labels**.

- Three strategies exist, ordered by quality vs. cost:

```python
┌─────────────────────────┬───────────┬───────────┬──────────────┐
│ Strategy                │ Quality   │ Cost      │ Scalability  │
├─────────────────────────┼───────────┼───────────┼──────────────┤
│ Human annotation        │ Highest   │ Very high │ Low          │
│ (PRM800K)               │           │           │              │
├─────────────────────────┼───────────┼───────────┼──────────────┤
│ LLM-as-a-judge          │ High      │ Medium    │ Medium       │
│ (Qwen2.5-72B as judge)  │           │           │              │
├─────────────────────────┼───────────┼───────────┼──────────────┤
│ Monte Carlo estimation  │ Medium    │ Low       │ High         │
│ (Math-Shepherd)         │           │           │              │
└─────────────────────────┴───────────┴───────────┴──────────────┘
```

In [12]:
@dataclass
class PRMStep:
    """A single reasoning step with its correctness label."""

    text: str
    label: int
    is_key: bool = False


@dataclass
class PRMExample:
    """
    A single PRM training example.
    Contains a problem and its full step-by-step solution, where each step has a correctness label.
    """

    problem: str
    steps: List[PRMStep]

    @property
    def num_steps(self) -> int:
        return len(self.steps)

    @property
    def labels(self) -> List[int]:
        return [s.label for s in self.steps]

    @property
    def is_correct_solution(self) -> bool:
        return all(s.label == 1 for s in self.steps)

## 3.4 Annotation Strategy

### 3.4.1 Human-Annotated (PRM800K style)

Lightman et al. (2023) hired human annotators to label each step as:
    - ✓ (correct): step is mathematically valid and logically sound
    - ✗ (incorrect): step contains an error
    - ? (neutral): step is valid but doesn't help

PRM800K dataset: ~800,000 step-level labels across ~75,000 problems.

Key insight: **supervise up to the first incorrect step only**.

Later steps are also labeled incorrect (contaminated by earlier error).

In [13]:
HUMAN_ANNOTATED_EXAMPLES: List[PRMExample] = [
    PRMExample(
        problem="A store had 50 apples. Sold 30% Monday and 20% of remainder Tuesday. How many left?",
        steps=[
            PRMStep("Monday sales = 30% of 50 = 0.30 × 50 = 15 apples.", label=1),
            PRMStep("After Monday: 50 − 15 = 35 apples remaining.", label=1),
            PRMStep("Tuesday sales = 20% of 35 = 0.20 × 35 = 7 apples.", label=1),
            PRMStep("After Tuesday: 35 − 7 = 28 apples left.", label=1),
        ],
    ),
    PRMExample(
        problem="A store had 50 apples. Sold 30% Monday and 20% of remainder Tuesday. How many left?",
        steps=[
            PRMStep("Monday sales = 30% of 50 = 15.", label=1),
            # ERROR: should apply 20% to REMAINDER (35), not original 50
            PRMStep("Tuesday sales = 20% of 50 = 10.", label=0, is_key=True),
            PRMStep("Total sold = 15 + 10 = 25.", label=0),
            PRMStep("Remaining = 50 − 25 = 25 apples.", label=0),
        ],
    ),
]

print(f"Human-annotated PRM examples: {len(HUMAN_ANNOTATED_EXAMPLES)}")
print(
    f"  Correct solutions:   {sum(1 for e in HUMAN_ANNOTATED_EXAMPLES if e.is_correct_solution)}"
)
print(
    f"  Incorrect solutions: {sum(1 for e in HUMAN_ANNOTATED_EXAMPLES if not e.is_correct_solution)}"
)

Human-annotated PRM examples: 2
  Correct solutions:   1
  Incorrect solutions: 1


### 3.4.2: Monte Carlo Estimation (Math-Shepherd)

The key insight of Math-Shepherd (Wang et al. 2023):
> **A step is "good" if rolling out from it can still reach the correct answer.**

Process:
1. For a given step, imagine you try to complete the rest of the solution M times (e.g., 16 attempts) starting from that step.
2. If at least one completion reaches the correct final answer, the step is probably fine (hard_label=1).
3. If zero completions succeed, the step is likely wrong (hard_label=0).
4. The soft_label gives you a probability (e.g., 12/16 = 0.75).

In [14]:
def extract_answer(text: str) -> str:
    """Extract the final numerical answer from a completion."""
    # Prefer an explicit "Final answer: <n>" line (markdown bold allowed)
    match = re.search(
        r"final answer\s*[:=]?\s*\**\s*\$?([\d,]+(?:\.\d+)?)", text, re.IGNORECASE
    )
    if match:
        return match.group(1).replace(",", "")

    # Fallback: last number in the completion
    nums = re.findall(r"\d+(?:\.\d+)?", text.replace(",", ""))
    return nums[-1] if nums else ""

In [15]:
def mc_complete(
    problem: str,
    step_prefix: List[str],
    gold_answer: str,
    M: int = 10,
    model: str = "gpt-4o-mini",
) -> Tuple[int, float]:
    """
    Monte Carlo estimation: given a problem and partial steps,
    ask the LLM to complete the solution M times.
    Check how many completions reach the correct final answer.

    Returns:
        hard_label: 1 if at least one completion is correct, else 0
        soft_label: fraction of correct completions (successes / M)
    """

    prefix_text = "\n".join(step_prefix)
    prompt = (
        f"Problem: {problem}\n\n"
        f"Solution so far:\n{prefix_text}\n\n"
        f"Continue solving from where the solution left off. "
        f"Show your work step by step and end with a line of the form "
        f"'Final answer: <number>'."
    )

    successes = 0

    # Generate M completions
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        n=M,  # generate M completions in one API call
        temperature=0.7,
        max_tokens=200,
    )

    for choice in response.choices:
        completion = choice.message.content
        extracted = extract_answer(completion)
        if extracted == gold_answer:
            successes += 1

    hard_label = int(successes > 0)
    soft_label = successes / M
    return hard_label, soft_label


def auto_annotate_mc(
    problem: str,
    steps: List[str],
    gold_answer: str,
    M: int = 8,
) -> "PRMExample":
    """
    Walk through each step, run MC completions from that prefix.
    Once first error is found, mark all subsequent steps as wrong.
    """
    annotated_steps = []
    found_error = False

    for k, step_text in enumerate(steps):

        # If found_error, subsequent step will all be marked as 0
        if found_error:
            annotated_steps.append(PRMStep(text=step_text, label=0))
            continue

        prefix = steps[: k + 1]
        hard_label, soft_label = mc_complete(problem, prefix, gold_answer, M=M)

        print(
            f"  Step {k+1}: MC success rate = {soft_label:.0%} | hard_label={hard_label}"
        )

        if hard_label == 0:
            found_error = True
            annotated_steps.append(PRMStep(text=step_text, label=0, is_key=True))
        else:
            annotated_steps.append(PRMStep(text=step_text, label=1))

    return PRMExample(problem=problem, steps=annotated_steps)


mc_problem = "A store had 50 apples. Sold 30% Monday and 20% of remainder Tuesday. How many left?"
correct_steps = [
    "Monday sales = 30% of 50 = 15.",
    "After Monday: 50 − 15 = 35 remaining.",
    "Tuesday sales = 20% of 35 = 7.",
    "After Tuesday: 35 − 7 = 28 left.",
]
wrong_steps = [
    "Monday sales = 30% of 50 = 15.",
    "Tuesday sales = 20% of 50 = 10.",
    "Total sold = 25.",
    "Remaining = 25.",
]

print("Annotating correct solution...")
mc_correct = auto_annotate_mc(mc_problem, correct_steps, "28", M=8)

print("\nAnnotating wrong solution...")
mc_wrong = auto_annotate_mc(mc_problem, wrong_steps, "28", M=8)

print("\n--- Correct solution MC labels ---")
for i, step in enumerate(mc_correct.steps):
    print(f"  Step {i+1}: label={step.label} | {step.text}")

print("\n--- Incorrect solution MC labels ---")
for i, step in enumerate(mc_wrong.steps):
    tag = " ← FIRST ERROR" if step.is_key else ""
    print(f"  Step {i+1}: label={step.label}{tag} | {step.text}")

Annotating correct solution...


  Step 1: MC success rate = 38% | hard_label=1


  Step 2: MC success rate = 100% | hard_label=1


  Step 3: MC success rate = 88% | hard_label=1


  Step 4: MC success rate = 75% | hard_label=1

Annotating wrong solution...


  Step 1: MC success rate = 62% | hard_label=1


  Step 2: MC success rate = 38% | hard_label=1


  Step 3: MC success rate = 25% | hard_label=1


  Step 4: MC success rate = 25% | hard_label=1

--- Correct solution MC labels ---
  Step 1: label=1 | Monday sales = 30% of 50 = 15.
  Step 2: label=1 | After Monday: 50 − 15 = 35 remaining.
  Step 3: label=1 | Tuesday sales = 20% of 35 = 7.
  Step 4: label=1 | After Tuesday: 35 − 7 = 28 left.

--- Incorrect solution MC labels ---
  Step 1: label=1 | Monday sales = 30% of 50 = 15.
  Step 2: label=1 | Tuesday sales = 20% of 50 = 10.
  Step 3: label=1 | Total sold = 25.
  Step 4: label=1 | Remaining = 25.


With a strong completer the wrong step often keeps `hard_label = 1`: the model notices the
mistake and finishes with the right answer anyway, so the rollout no longer measures the
step. Math-Shepherd used a weaker completer than the policy being judged, and its soft labels
(success rate) carry more information than the hard 0/1 label.

### 3.4.3 LLM-as-a-Judge

Use a strong LLM to evaluate each step.

Prompt the judge with the problem, prior steps, and the current step, and ask it to classify the step as correct or incorrect.

In [16]:
LLM_JUDGE_PROMPT_TEMPLATE = """
You are a mathematics verification expert.

Problem: {problem}

Previous steps:
{prev_steps}

Current step to evaluate:
{current_step}

Is this step mathematically correct given the problem and previous steps?
Answer with only: CORRECT or INCORRECT
Reason: (brief explanation)
"""


def llm_judge(prompt: str, model: str = "gpt-4o-mini") -> str:
    """Call OpenAI API to judge whether a reasoning step is correct."""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,  # deterministic for consistent judgments
        max_tokens=100,
    )
    return response.choices[0].message.content


def llm_judge_annotation(
    problem: str,
    steps: List[str],
    model: str = "gpt-4o-mini",
) -> "PRMExample":
    """
    Walk through each step and ask the LLM judge if it's correct.
    Once first error is found, mark all subsequent steps as wrong.
    """
    annotated = []
    found_error = False

    for k, step in enumerate(steps):
        if found_error:
            annotated.append(PRMStep(text=step, label=0))
            continue

        prev_steps_str = (
            "\n".join(f"Step {i+1}: {s}" for i, s in enumerate(steps[:k]))
            if k > 0
            else "(none)"
        )
        prompt = LLM_JUDGE_PROMPT_TEMPLATE.format(
            problem=problem,
            prev_steps=prev_steps_str,
            current_step=f"Step {k+1}: {step}",
        )

        verdict = llm_judge(prompt, model=model)
        # Check INCORRECT explicitly: "CORRECT" is a substring of "INCORRECT"
        label = 0 if verdict.strip().upper().startswith("INCORRECT") else 1

        print(f"  Step {k+1}: {verdict.strip()}")

        if label == 0:
            found_error = True
            annotated.append(PRMStep(text=step, label=0, is_key=True))
        else:
            annotated.append(PRMStep(text=step, label=1))

    return PRMExample(problem=problem, steps=annotated)


# === Test Run ===
mc_problem = "A store had 50 apples. Sold 30% Monday and 20% of remainder Tuesday. How many left?"

correct_steps = [
    "Monday sales = 30% of 50 = 15.",
    "After Monday: 50 − 15 = 35 remaining.",
    "Tuesday sales = 20% of 35 = 7.",
    "After Tuesday: 35 − 7 = 28 left.",
]
wrong_steps = [
    "Monday sales = 30% of 50 = 15.",
    # ERROR: should be 20% of 35
    "Tuesday sales = 20% of 50 = 10.",
    "Total sold = 25.",
    "Remaining = 25.",
]

print("LLM Judge on correct solution:")
llm_correct = llm_judge_annotation(mc_problem, correct_steps)

print("\nLLM Judge on wrong solution:")
llm_wrong = llm_judge_annotation(mc_problem, wrong_steps)

print("\n--- Correct solution labels ---")
for i, step in enumerate(llm_correct.steps):
    print(f"  Step {i+1}: label={step.label} | {step.text}")

print("\n--- Incorrect solution labels ---")
for i, step in enumerate(llm_wrong.steps):
    tag = " ← FIRST ERROR" if step.is_key else ""
    print(f"  Step {i+1}: label={step.label}{tag} | {step.text}")

LLM Judge on correct solution:


  Step 1: CORRECT  
Reason: 30% of 50 apples is calculated as 0.30 * 50 = 15, which is accurate.


  Step 2: CORRECT  
Reason: The calculation of the remaining apples after selling 15 apples from the initial 50 is accurate: 50 - 15 = 35.


  Step 3: CORRECT  
Reason: The calculation of Tuesday sales as 20% of the remaining 35 apples is accurate, as 20% of 35 equals 7.


  Step 4: CORRECT  
Reason: The calculation of remaining apples after Tuesday is accurate; 35 apples minus 7 apples sold on Tuesday equals 28 apples left.

LLM Judge on wrong solution:


  Step 1: CORRECT  
Reason: 30% of 50 apples is calculated as 0.30 * 50 = 15 apples, which is accurate.


  Step 2: INCORRECT  
Reason: The Tuesday sales should be calculated as 20% of the remainder after Monday's sales, not 20% of the original 50 apples. The remainder after Monday is 50 - 15 = 35 apples, so Tuesday sales should be 20% of 35, which is 7 apples.

--- Correct solution labels ---
  Step 1: label=1 | Monday sales = 30% of 50 = 15.
  Step 2: label=1 | After Monday: 50 − 15 = 35 remaining.
  Step 3: label=1 | Tuesday sales = 20% of 35 = 7.
  Step 4: label=1 | After Tuesday: 35 − 7 = 28 left.

--- Incorrect solution labels ---
  Step 1: label=1 | Monday sales = 30% of 50 = 15.
  Step 2: label=0 ← FIRST ERROR | Tuesday sales = 20% of 50 = 10.
  Step 3: label=0 | Total sold = 25.
  Step 4: label=0 | Remaining = 25.


## 3.5 PRM Dataset and Tokenization

**The Step Token Trick**

The PRM needs to know WHERE in the token sequence each step ends.
The approach from Math-Shepherd and PRM800K:

1. Mark where each step ends: a special [STEP] token, or (as below) a newline after each step,
   with the step's last token located from the tokenizer's character offsets
2. Record the token position of each step end
3. During forward pass: extract hidden state at [STEP] positions
4. Apply step head → step scores
5. Compute BCE loss at step positions ONLY (rest masked)

Input sequence:
    [CLS] problem text [SEP] step_1_text ▪ step_2_text ▪ step_3_text ▪
                                          ↑              ↑              ↑
                                       pos 15          pos 28         pos 41
                                    label=1          label=1         label=0

In [17]:
def solution_features(problem_list: List[str], max_solutions: int):
    """Padded per-step features, labels and mask for up to max_solutions solutions."""
    feats, labels = [], []
    for p in problem_list:
        for steps, step_labels in solutions[p]:
            feats.append(encode_solution(p, steps)[1])
            labels.append(torch.tensor(step_labels, dtype=torch.float))
            if len(feats) == max_solutions:
                break
        if len(feats) == max_solutions:
            break
    max_steps = max(f.size(0) for f in feats)
    # (num_solutions, max_steps, hidden_dim), (num_solutions, max_steps) x 2
    X = torch.zeros(len(feats), max_steps, HIDDEN_DIM)
    y = torch.zeros(len(feats), max_steps)
    mask = torch.zeros(len(feats), max_steps)
    for i, (f, l) in enumerate(zip(feats, labels)):
        X[i, : f.size(0)], y[i, : l.size(0)], mask[i, : l.size(0)] = f, l, 1.0
    return X, y, mask


start = time.time()
prm_train = solution_features(train_problems, 1200)
prm_test = solution_features(test_problems, 300)
num_pos = (prm_train[1] * prm_train[2]).sum()
num_neg = prm_train[2].sum() - num_pos
print(
    f"PRM solutions: {prm_train[0].size(0)} train, {prm_test[0].size(0)} test "
    f"({time.time() - start:.0f}s to encode)"
)
print(f"Train steps: {int(num_pos)} correct, {int(num_neg)} wrong")

PRM solutions: 1200 train, 300 test (221s to encode)
Train steps: 1948 correct, 2384 wrong


## 3.6 Training


In [18]:
def train_prm(
    model: ProcessRewardModel,
    train_data: Tuple[torch.Tensor, torch.Tensor, torch.Tensor],
    test_data: Tuple[torch.Tensor, torch.Tensor, torch.Tensor],
    num_epochs: int = 60,
    lr: float = 1e-3,
    batch_num: int = 64,
) -> Tuple[List[float], List[float], List[float]]:
    """Train only the step head on cached step features, class-balanced BCE."""
    head = model.step_head
    optimizer = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=0.01)
    X, y, mask = (t.to(DEVICE) for t in train_data)
    X_test, y_test, mask_test = (t.to(DEVICE) for t in test_data)
    # Balance the classes: weight correct steps by (#wrong / #correct)
    pos = (y * mask).sum()
    pos_weight = (mask.sum() - pos) / pos

    def step_accuracy(X_, y_, mask_) -> Tuple[float, float]:
        with torch.no_grad():
            pred = (head(X_).squeeze(-1) > 0).float()
        valid = mask_.bool()
        acc_correct = (pred[valid & (y_ == 1)] == 1).float().mean().item()
        acc_wrong = (pred[valid & (y_ == 0)] == 0).float().mean().item()
        return acc_correct, acc_wrong

    loss_history, acc_correct_hist, acc_wrong_hist = [], [], []
    for epoch in range(num_epochs):
        head.train()
        perm = torch.randperm(X.size(0), device=DEVICE)
        epoch_losses = []
        for i in range(0, len(perm), batch_num):
            idx = perm[i : i + batch_num]
            # (batch_num, max_steps, hidden_dim) → (batch_num, max_steps)
            step_scores = head(X[idx]).squeeze(-1)
            loss = prm_loss(step_scores, y[idx], mask[idx], pos_weight=pos_weight)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())

        head.eval()
        acc_c, acc_w = step_accuracy(X_test, y_test, mask_test)
        loss_history.append(np.mean(epoch_losses))
        acc_correct_hist.append(acc_c)
        acc_wrong_hist.append(acc_w)
        if epoch % 10 == 0 or epoch == num_epochs - 1:
            print(
                f"  Epoch {epoch+1:3d}/{num_epochs} | Loss: {loss_history[-1]:.4f} | "
                f"test Acc(correct steps): {acc_c:.1%} | test Acc(wrong steps): {acc_w:.1%}"
            )
    return loss_history, acc_correct_hist, acc_wrong_hist


# === Test Run ===
prm_losses, prm_acc_c, prm_acc_w = train_prm(prm, prm_train, prm_test)

  Epoch   1/60 | Loss: 0.9099 | test Acc(correct steps): 61.0% | test Acc(wrong steps): 68.5%


  Epoch  11/60 | Loss: 0.3491 | test Acc(correct steps): 60.8% | test Acc(wrong steps): 70.8%


  Epoch  21/60 | Loss: 0.2121 | test Acc(correct steps): 70.9% | test Acc(wrong steps): 63.4%


  Epoch  31/60 | Loss: 0.1190 | test Acc(correct steps): 55.9% | test Acc(wrong steps): 73.2%


  Epoch  41/60 | Loss: 0.1051 | test Acc(correct steps): 71.1% | test Acc(wrong steps): 63.9%


  Epoch  51/60 | Loss: 0.0483 | test Acc(correct steps): 61.0% | test Acc(wrong steps): 70.8%


  Epoch  60/60 | Loss: 0.0381 | test Acc(correct steps): 69.6% | test Acc(wrong steps): 65.3%


## 3.7 Inference

**PRM at Inference Time**

The PRM has two main uses at inference:

**1. Step-level scoring** (find where errors occur):
    Score each step and identify the first step that drops below a threshold.

**2. Best-of-N with PRM aggregation** (solution selection):
    Score each candidate solution and aggregate step scores.

Two common aggregation strategies:
- **Min-score**: r_solution = min(r_1, ..., r_K)
  Conservative: a solution is only as good as its weakest step.
  Used in Lightman et al. (2023).

- **Product**: r_solution = Π_{k=1}^K r_k
  Penalizes multiple low-confidence steps.
  Equivalent to summing log-probabilities.

- **Last-step**: r_solution = r_K
  Uses only the final step's score.
  Degenerates toward ORM behavior — generally avoided.

In [19]:
@torch.no_grad()
def prm_score_steps(
    model: ProcessRewardModel,
    problem: str,
    steps: List[str],
    tokenizer,
    max_len: int = 512,
) -> List[float]:
    """
    Score each step in a solution with the PRM.
    Returns a probability in [0, 1] for each step (1 = correct).
    """
    model.eval()
    # (num_steps, hidden_dim): hidden state at the last token of every step
    _, step_feats = encode_solution(problem, steps, max_len)
    # (num_steps, hidden_dim) → (num_steps,)
    return (
        torch.sigmoid(model.step_head(step_feats.to(DEVICE)).squeeze(-1)).cpu().tolist()
    )


def best_of_n_prm(
    model: ProcessRewardModel,
    problem: str,
    solution_steps: List[List[str]],
    tokenizer,
    aggregation: str = "min",
    verbose: bool = True,
) -> Tuple[int, float, List[float]]:
    """
    Best-of-N selection using PRM scoring.
    Scores each candidate solution by aggregating its step scores, then returns the index of the best solution.
    """
    all_solution_scores = []

    for sol_idx, steps in enumerate(solution_steps):
        step_probs = prm_score_steps(model, problem, steps, tokenizer)

        if aggregation == "min":
            sol_score = min(step_probs) if step_probs else 0.0
        elif aggregation == "product":
            sol_score = math.prod(step_probs) if step_probs else 0.0
        elif aggregation == "mean":
            sol_score = sum(step_probs) / len(step_probs) if step_probs else 0.0
        elif aggregation == "last":
            sol_score = step_probs[-1] if step_probs else 0.0
        else:
            raise ValueError(f"Unknown aggregation: {aggregation}")

        all_solution_scores.append(sol_score)

        if verbose:
            print(f"  Solution {sol_idx+1}: aggregated={sol_score:.4f}")
            for k, (step, prob) in enumerate(zip(steps, step_probs)):
                flag = "x" if prob < 0.5 else "✓"
                print(f"    Step {k+1}: p={prob:.3f} {flag} | {step[:50]}")

    best_idx = int(np.argmax(all_solution_scores))
    best_score = all_solution_scores[best_idx]
    return best_idx, best_score, all_solution_scores


# Demo PRM inference
mc_problem = "A store had 50 apples. Sold 30% Monday and 20% of remainder Tuesday. How many left?"
solutions_as_steps = [
    [
        "Step 1: Monday sales = 30% of 50 = 15.",
        "Step 2: After Monday: 50 − 15 = 35.",
        "Step 3: Tuesday sales = 20% of 35 = 7.",
        "Step 4: After Tuesday: 35 − 7 = 28. The answer is: 28",
    ],
    [
        "Step 1: Monday sales = 30% of 50 = 15.",
        "Step 2: Tuesday sales = 20% of 50 = 10.",  # ERROR
        "Step 3: Total sold = 25.",
        "Step 4: Remaining = 25. The answer is: 25",
    ],
]

best_idx, best_score, all_scores = best_of_n_prm(
    prm, mc_problem, solutions_as_steps, tokenizer, aggregation="min", verbose=True
)
print(f"\nBest solution index: {best_idx+1} (score={best_score:.4f})")

  Solution 1: aggregated=0.9988
    Step 1: p=1.000 ✓ | Step 1: Monday sales = 30% of 50 = 15.
    Step 2: p=1.000 ✓ | Step 2: After Monday: 50 − 15 = 35.
    Step 3: p=0.999 ✓ | Step 3: Tuesday sales = 20% of 35 = 7.
    Step 4: p=0.999 ✓ | Step 4: After Tuesday: 35 − 7 = 28. The answer is:
  Solution 2: aggregated=0.3735
    Step 1: p=1.000 ✓ | Step 1: Monday sales = 30% of 50 = 15.
    Step 2: p=0.374 x | Step 2: Tuesday sales = 20% of 50 = 10.
    Step 3: p=0.952 ✓ | Step 3: Total sold = 25.
    Step 4: p=0.990 ✓ | Step 4: Remaining = 25. The answer is: 25

Best solution index: 1 (score=0.9988)


## Summary Table

**ORM limitations:**
1. Cannot identify WHERE in the reasoning chain an error occurred
2. Incorrect solutions with many correct steps get same reward as solutions that are wrong from step 1 — poor credit assignment
3. In RL, sparse terminal reward makes long-horizon training unstable

**PRM limitations:**
1. Annotation bottleneck: MC estimation is noisy, human annotation is expensive
2. "Reward hacking": models learn to score high on individual steps while still producing a wrong final answer
3. Qwen (2025) found many PRMs degrade to ORM-like behavior — they assign low scores to the FINAL step containing the answer, rather than the intermediate step where the error actually occurred
4. Step granularity is ambiguous — what counts as one "step"?
5. Difficult to apply to open-ended tasks (no clear "correct" label per step)